## Robust Regression Engine 

**Import Libraries**

In [1]:
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
from sklearn.linear_model import Lasso,Ridge
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import TimeSeriesSplit
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

**Import Dataset**

In [2]:
df = pd.read_csv("Real Estate Data.csv")

print(df.head())

   property_id   sale_date  area_sqft  bedrooms  bathrooms  location_score  \
0       200001  2014-01-01       2181         6          4             8.1   
1       200002  2019-12-01       2383         5          4             5.3   
2       200003  2016-10-01       1047         3          3             5.9   
3       200004  2013-03-01       1753         4          3             7.0   
4       200005  2013-07-01       1728         4          4            10.0   

   property_age  distance_city_km  near_school  near_metro  crime_rate_index  \
0            21               3.8            0           0              4.84   
1            28              10.9            1           1              2.89   
2             7              27.5            0           1              4.04   
3            27              12.1            0           0              3.28   
4            32               1.4            0           1              3.84   

   house_price_inr  
0         35154898  
1       

## Part B — Dataset Understanding & Preparation

**Task 6 — Identify Features and Target**

In [3]:
# Features

x = df[
    [
    "property_id",
    "area_sqft",
    "bedrooms",
    "bathrooms",
    "location_score",
    "property_age",
    "distance_city_km",
    "near_school",
    "near_metro",
    "crime_rate_index"
    ]
]

# Target 

y = df["house_price_inr"]

print("\nFeatures")
print(x.head())

print("\nTarget")
print(y.head())


Features
   property_id  area_sqft  bedrooms  bathrooms  location_score  property_age  \
0       200001       2181         6          4             8.1            21   
1       200002       2383         5          4             5.3            28   
2       200003       1047         3          3             5.9             7   
3       200004       1753         4          3             7.0            27   
4       200005       1728         4          4            10.0            32   

   distance_city_km  near_school  near_metro  crime_rate_index  
0               3.8            0           0              4.84  
1              10.9            1           1              2.89  
2              27.5            0           1              4.04  
3              12.1            0           0              3.28  
4               1.4            0           1              3.84  

Target
0    35154898
1    26710893
2    11216242
3    21984310
4    25080429
Name: house_price_inr, dtype: int64


**Task 7 — Train-Test Split**

In [4]:
# Split data

x_train,x_test,y_train,y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data:", x_train.shape)
print("Testing Data:", x_test.shape)

Training Data: (3040, 10)
Testing Data: (760, 10)


**Task 8 — Basic Preprocessing / Scaling**

In [5]:
# Create Scaler
scaler = StandardScaler()

# Fit and transform training data
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.fit_transform(x_test)

print("Scaling completed.")



Scaling completed.


## Part C — Regularized Linear Models

**Task 9 — Ridge Regression (L2)**

In [6]:
# Create Ridge Model
ridge_model = Ridge(alpha=1.0)

# Train model
ridge_model.fit(x_train_scaled, y_train)

# Predictions

ridge_train_pred = ridge_model.predict(x_train_scaled)
ridge_test_pred = ridge_model.predict(x_test_scaled)

# Calculate errors

ridge_train_mse = mean_squared_error(y_train, ridge_train_pred)
ridge_test_mse = mean_squared_error(y_test, ridge_test_pred)

# Calculate R2

ridge_train_R2 = r2_score(y_train,ridge_train_pred)
ridge_test_r2 = r2_score(y_test, ridge_test_pred)

print("Ridge Regression")

print("Training MSE:", ridge_test_mse)
print("Test MSE:", ridge_test_mse)

print("Training R2:", ridge_train_R2)
print("Test R2:", ridge_test_r2)

Ridge Regression
Training MSE: 6608582956270.712
Test MSE: 6608582956270.712
Training R2: 0.9162466892236396
Test R2: 0.9179416519226019


**Task 10 — Lasso Regression (L1)**

In [7]:
# Create Lasso Model 
lasso_model = Lasso(alpha=1.0)

# Train Model

lasso_model.fit(x_train_scaled, y_train)

# Predictios 

lasso_train_pred = lasso_model.predict(x_train_scaled)
lasso_test_pred = lasso_model.predict(x_test_scaled)

# Calculate errors

lasso_train_mse = mean_squared_error(y_train,lasso_train_pred)
lasso_test_mse = mean_squared_error(y_test,lasso_test_pred)

# Calculate R2

lasso_train_r2 = r2_score(y_train,lasso_train_pred)
lasso_test_r2 = r2_score(y_test, lasso_test_pred)

print("Lasso Regression")

print("\nTraining MSE:", lasso_train_mse)
print("\nTest MSE:", lasso_test_mse)

print("\nTraining R2:",lasso_train_r2)
print("\nTest R2:", lasso_test_r2)

Lasso Regression

Training MSE: 6248456088728.442

Test MSE: 6606879657113.879

Training R2: 0.9162469410906897

Test R2: 0.9179628016783083


**Task 11 — Tune Alpha Using Cross-Validation**

In [8]:
alphas = [0.01, 0.1, 1, 10, 100]

print("Ridge Cross-Validation Results")

ridge_results = []

for alpha in alphas:

    model = Ridge(alpha=alpha)

    scores = cross_val_score(
        model,
        x_train_scaled,
        y_train,
        cv=5,
        scoring="neg_mean_squared_error"
    )

    mse = -scores.mean()

    ridge_results.append(mse)

    print("Alpha:", alpha, "Mean MSE:", mse)


print("\nLasso Cross-Validation Results")

lasso_results = []

for alpha in alphas:

    model = Lasso(alpha=alpha)

    scores = cross_val_score(
        model,
        x_train_scaled,
        y_train,
        cv=5,
        scoring="neg_mean_squared_error"
    )

    mse = -scores.mean()

    lasso_results.append(mse)

    print("Alpha:", alpha, "Mean MSE:", mse)

    best_ridge_alpha = alphas[np.argmin(ridge_results)]

best_lasso_alpha = alphas[np.argmin(lasso_results)]

print("\nBest Ridge Alpha:", best_ridge_alpha)
print("Best Lasso Alpha:", best_lasso_alpha)

Ridge Cross-Validation Results
Alpha: 0.01 Mean MSE: 6307675329824.247
Alpha: 0.1 Mean MSE: 6307670550098.521
Alpha: 1 Mean MSE: 6307649247927.768
Alpha: 10 Mean MSE: 6309975446387.802
Alpha: 100 Mean MSE: 6509611161437.586

Lasso Cross-Validation Results
Alpha: 0.01 Mean MSE: 6307675886978.262
Alpha: 0.1 Mean MSE: 6307675853102.213
Alpha: 1 Mean MSE: 6307675512644.993
Alpha: 10 Mean MSE: 6307672121727.114
Alpha: 100 Mean MSE: 6307638254493.283

Best Ridge Alpha: 1
Best Lasso Alpha: 100


**Task 12 — Compare Ridge and Lasso**

In [9]:
# Train final Ridge model
final_ridge = Ridge(alpha=best_ridge_alpha)
final_ridge.fit(x_train_scaled, y_train)

# Train final Lasso model
final_lasso = Lasso(alpha=best_lasso_alpha)
final_lasso.fit(x_train_scaled, y_train)


# Predictions
ridge_pred = final_ridge.predict(x_test_scaled)
lasso_pred = final_lasso.predict(x_test_scaled)


# MSE
ridge_mse = mean_squared_error(y_test, ridge_pred)
lasso_mse = mean_squared_error(y_test, lasso_pred)


# R2
ridge_r2 = r2_score(y_test, ridge_pred)
lasso_r2 = r2_score(y_test, lasso_pred)


print("----- Final Comparison -----")

print("\nRidge")
print("Test MSE:", ridge_mse)
print("Test R2:", ridge_r2)

print("\nLasso")
print("Test MSE:", lasso_mse)
print("Test R2:", lasso_r2)

----- Final Comparison -----

Ridge
Test MSE: 6608582956270.712
Test R2: 0.9179416519226019

Lasso
Test MSE: 6607014976518.959
Test R2: 0.9179611214259885


## Part D — Cross-Validation Strategies

**Task 13 — K-Fold Cross-Validation**

In [10]:
# K-Fold
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Model
ridge = Ridge(alpha=1.0)

# Cross-validation
scores = cross_val_score(
    ridge,
    x_train_scaled,
    y_train,
    cv=kf,
    scoring="neg_mean_squared_error"
)

# Convert negative MSE to positive
mse_scores = -scores

print("K-Fold MSE Scores:")
print(mse_scores)

print("\nMean MSE:", mse_scores.mean())

K-Fold MSE Scores:
[6.30473304e+12 7.13516295e+12 6.08100328e+12 5.82494332e+12
 6.21630011e+12]

Mean MSE: 6312428540249.873


**Task 13 — Stratified K-Fold by Binning Target**

In [11]:

# Create price bins
y_bins = pd.qcut(
    y_train,
    q=5,
    labels=False,
    duplicates="drop"
)

# Stratified K-Fold
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_index, val_index in skf.split(x_train_scaled, y_bins):

    X_tr = x_train_scaled[train_index]
    X_val = x_train_scaled[val_index]

    y_tr = y_train.iloc[train_index]
    y_val = y_train.iloc[val_index]

    model = Ridge(alpha=1.0)

    model.fit(X_tr, y_tr)

    predictions = model.predict(X_val)

    mse = ((y_val - predictions) ** 2).mean()

    scores.append(mse)

print("Stratified K-Fold MSE:")
print(scores)

print("\nMean MSE:", sum(scores) / len(scores))

Stratified K-Fold MSE:
[np.float64(6577013463781.584), np.float64(6037709314080.801), np.float64(5968144380003.151), np.float64(5960704335184.828), np.float64(6951796504559.57)]

Mean MSE: 6299073599521.986


**Task 13 — Leave-One-Out Cross-Validation**

In [12]:
loo = LeaveOneOut()

errors = []

for train_index, val_index in loo.split(x_train_scaled):

    X_tr = x_train_scaled[train_index]
    X_val = x_train_scaled[val_index]

    y_tr = y_train.iloc[train_index]
    y_val = y_train.iloc[val_index]

    model = Ridge(alpha=1.0)

    model.fit(X_tr, y_tr)

    prediction = model.predict(X_val)

    error = (y_val.iloc[0] - prediction[0]) ** 2

    errors.append(error)

print("LOOCV Mean MSE:", sum(errors) / len(errors))

LOOCV Mean MSE: 6297880307909.75


**Task 13 — Time Series Split**

In [13]:
df["sale_date"] = pd.to_datetime(df["sale_date"])

df = df.sort_values("sale_date")

print(df[["sale_date", "house_price_inr"]].head())

      sale_date  house_price_inr
3091 2010-01-01         37038561
1609 2010-01-01         16745091
1420 2010-01-01         31164056
2024 2010-01-01         32351766
529  2010-01-01         13755440


In [14]:
# # Remove Date from model features
# X_time =  df[
#     [
#     "property_id",
#     "sale_date",
#     "area_sqft",
#     "bedrooms",
#     "bathrooms",
#     "location_score",
#     "property_age",
#     "distance_city_km",
#     "near_school",
#     "near_metro",
#     "crime_rate_index"
#     ]
# ]

# y_time = df["house_price_inr"]

# # Check that all features are numeric
# print(X_time.dtypes)

# # Scale only numeric features
# scaler_time = StandardScaler()

# X_time_scaled = scaler_time.fit_transform(X_time)

# # Time Series Split
# tscv = TimeSeriesSplit(n_splits=5)

# scores = []

# for train_index, test_index in tscv.split(X_time_scaled):

#     X_tr = X_time_scaled[train_index]
#     X_val = X_time_scaled[test_index]

#     y_tr = y_time.iloc[train_index]
#     y_val = y_time.iloc[test_index]

#     # Ridge model
#     model = Ridge(alpha=1.0)

#     # Train
#     model.fit(X_tr, y_tr)

#     # Prediction
#     prediction = model.predict(X_val)

#     # MSE
#     mse = mean_squared_error(y_val, prediction)

#     scores.append(mse)

# print("\nTime Series MSE:")
# print(scores)

# print("\nMean MSE:", np.mean(scores))

**Task 14 — Compare Cross-Validation Strategies**

K-Fold
→ Randomly shuffled folds

Stratified K-Fold
→ Similar target-price groups maintain karta hai

LOOCV
→ One row at a time validation

Time Series Split
→ Past se future predict karta hai

## Part E — Tree-Based Regression

**Task 15 — Decision Tree Regression**

In [15]:
# 1. Select numeric features

X_tree = df.select_dtypes(include=np.number).copy()

# Remove target column
X_tree = X_tree.drop(columns=["house_price_inr"], errors="ignore")

# Target
y_tree = df["house_price_inr"]

# 2. Train-Test Split

x_train, x_test, y_train, y_test = train_test_split(
    X_tree,
    y_tree,
    test_size=0.20,
    random_state=42
)

# 3. Decision Tree

dt_model = DecisionTreeRegressor(
    max_depth=5,
    min_samples_leaf=5,
    random_state=42
)

# 4. Train

dt_model.fit(x_train, y_train)

# 5. Prediction

dt_pred = dt_model.predict(x_test)

# 6. Evaluation

dt_mse = mean_squared_error(y_test, dt_pred)
dt_mae = mean_absolute_error(y_test, dt_pred)
dt_rmse = np.sqrt(dt_mse)
dt_r2 = r2_score(y_test, dt_pred)


print("Decision Tree Regression")
print("------------------------")
print("MSE :", dt_mse)
print("MAE :", dt_mae)
print("RMSE:", dt_rmse)

Decision Tree Regression
------------------------
MSE : 9298331889057.13
MAE : 2346347.9235661468
RMSE: 3049316.626566866


**Task 16 — Control Tree Complexity**

In [16]:
dt_model = DecisionTreeRegressor(
    max_depth=5,
    min_samples_leaf=5,
    random_state=42
)

dt_model.fit(x_train, y_train)

dt_pred = dt_model.predict(x_test)

print("MSE:", mean_squared_error(y_test, dt_pred))
print("MAE:", mean_absolute_error(y_test, dt_pred))
print("R2:", r2_score(y_test, dt_pred))

MSE: 9298331889057.13
MAE: 2346347.9235661468
R2: 0.8788803090875084


**Task 17 — Random Forest Regression**

In [17]:
# Create Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42
)

# Train
rf_model.fit(x_train, y_train)

# Prediction
rf_pred = rf_model.predict(x_test)

# Metrics
rf_mse = mean_squared_error(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest")

print("MSE:", rf_mse)
print("MAE:", rf_mae)
print("R2:", rf_r2)

Random Forest
MSE: 5604728932825.642
MAE: 1772108.3098046228
R2: 0.926993030137906


**Task 18 — Single Tree vs Random Forest**

In [18]:
tree_results = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Random Forest"
    ],
    
    "MSE": [
        dt_mse,
        rf_mse
    ],
    
    "MAE": [
        dt_mae,
        rf_mae
    ],
    
    "R2": [
        dt_r2,
        rf_r2
    ]
})

print(tree_results)

           Model           MSE           MAE        R2
0  Decision Tree  9.298332e+12  2.346348e+06  0.878880
1  Random Forest  5.604729e+12  1.772108e+06  0.926993


## Part F — Support Vector Regression

**Task 19 — SVR Linear Kernel**

In [19]:
# Linear SVR
svr_linear = SVR(
    kernel="linear",
    C=1.0,
    epsilon=0.1
)

# Train
svr_linear.fit(x_train_scaled, y_train)

# Prediction
svr_linear_pred = svr_linear.predict(x_test_scaled)

print("Linear SVR")

print("MSE:",
      mean_squared_error(y_test, svr_linear_pred))

print("MAE:",
      mean_absolute_error(y_test, svr_linear_pred))

print("R2:",
      r2_score(y_test, svr_linear_pred))

Linear SVR
MSE: 76973610482091.17
MAE: 6872819.438308449
R2: -0.0026551021459384483


**Task 19 — SVR RBF Kernel**

In [20]:
# RBF SVR
svr_rbf = SVR(
    kernel="rbf",
    C=1.0,
    epsilon=0.1,
    gamma="scale"
)

# Train
svr_rbf.fit(x_train_scaled, y_train)

# Prediction
svr_rbf_pred = svr_rbf.predict(x_test_scaled)

print("RBF SVR")

print("MSE:",
      mean_squared_error(y_test, svr_rbf_pred))

print("MAE:",
      mean_absolute_error(y_test, svr_rbf_pred))

print("R2:",
      r2_score(y_test, svr_rbf_pred))

RBF SVR
MSE: 76973638310997.77
MAE: 6872801.450208486
R2: -0.002655464644137462


**Polynomial Kernel**

In [21]:
svr_poly = SVR(
    kernel="poly",
    C=1.0,
    epsilon=0.1,
    degree=3
)

svr_poly.fit(x_train_scaled, y_train)

svr_poly_pred = svr_poly.predict(x_test_scaled)

print("Polynomial SVR")

print("MSE:",
      mean_squared_error(y_test, svr_poly_pred))

print("MAE:",
      mean_absolute_error(y_test, svr_poly_pred))

print("R2:",
      r2_score(y_test, svr_poly_pred))

Polynomial SVR
MSE: 76973670503801.38
MAE: 6872802.5678082965
R2: -0.0026558839862806938


## Task 20 — Tune C, Gamma and Epsilon

In [22]:
C_values = [0.1, 1, 10]
gamma_values = ["scale", 0.01, 0.1]
epsilon_values = [0.01, 0.1, 0.5]

best_mse = float("inf")

best_C = None
best_gamma = None
best_epsilon = None

for C in C_values:

    for gamma in gamma_values:

        for epsilon in epsilon_values:

            model = SVR(
                kernel="rbf",
                C=C,
                gamma=gamma,
                epsilon=epsilon
            )

            model.fit(x_train_scaled, y_train)

            prediction = model.predict(x_test_scaled)

            mse = mean_squared_error(
                y_test,
                prediction
            )

            if mse < best_mse:

                best_mse = mse
                best_C = C
                best_gamma = gamma
                best_epsilon = epsilon


print("Best C:", best_C)
print("Best Gamma:", best_gamma)
print("Best Epsilon:", best_epsilon)
print("Best MSE:", best_mse)

Best C: 10
Best Gamma: scale
Best Epsilon: 0.01
Best MSE: 76973589631093.94


**Task 21 — Compare SVR with Linear and Tree Models**

In [23]:
results = pd.DataFrame({

    "Model": [
        "Ridge",
        "Lasso",
        "Decision Tree",
        "Random Forest",
        "SVR Linear",
        "SVR RBF",
        "SVR Polynomial"
    ],

    "MSE": [
        ridge_mse,
        lasso_mse,
        dt_mse,
        rf_mse,
        mean_squared_error(y_test, svr_linear_pred),
        mean_squared_error(y_test, svr_rbf_pred),
        mean_squared_error(y_test, svr_poly_pred)
    ],

    "MAE": [
        mean_absolute_error(y_test, ridge_pred),
        mean_absolute_error(y_test, lasso_pred),
        dt_mae,
        rf_mae,
        mean_absolute_error(y_test, svr_linear_pred),
        mean_absolute_error(y_test, svr_rbf_pred),
        mean_absolute_error(y_test, svr_poly_pred)
    ],

    "R2": [
        ridge_r2,
        lasso_r2,
        dt_r2,
        rf_r2,
        r2_score(y_test, svr_linear_pred),
        r2_score(y_test, svr_rbf_pred),
        r2_score(y_test, svr_poly_pred)
    ]
})

print(results)

            Model           MSE           MAE        R2
0           Ridge  6.608583e+12  9.243632e+06  0.917942
1           Lasso  6.607015e+12  9.244905e+06  0.917961
2   Decision Tree  9.298332e+12  2.346348e+06  0.878880
3   Random Forest  5.604729e+12  1.772108e+06  0.926993
4      SVR Linear  7.697361e+13  6.872819e+06 -0.002655
5         SVR RBF  7.697364e+13  6.872801e+06 -0.002655
6  SVR Polynomial  7.697367e+13  6.872803e+06 -0.002656


## Part G — Model Comparison & Evaluation

**Task 22 — MSE, MAE, RMSE, R²**

In [24]:
import numpy as np

results["RMSE"] = np.sqrt(results["MSE"])

print(results)

            Model           MSE           MAE        R2          RMSE
0           Ridge  6.608583e+12  9.243632e+06  0.917942  2.570716e+06
1           Lasso  6.607015e+12  9.244905e+06  0.917961  2.570411e+06
2   Decision Tree  9.298332e+12  2.346348e+06  0.878880  3.049317e+06
3   Random Forest  5.604729e+12  1.772108e+06  0.926993  2.367431e+06
4      SVR Linear  7.697361e+13  6.872819e+06 -0.002655  8.773461e+06
5         SVR RBF  7.697364e+13  6.872801e+06 -0.002655  8.773462e+06
6  SVR Polynomial  7.697367e+13  6.872803e+06 -0.002656  8.773464e+06


**Task 23 — Regularized vs Tree-Based Models**

In [25]:
comparison = pd.DataFrame({

    "Model Type": [
        "Regularized Linear",
        "Regularized Linear",
        "Tree-Based",
        "Tree-Based"
    ],

    "Model": [
        "Ridge",
        "Lasso",
        "Decision Tree",
        "Random Forest"
    ],

    "MSE": [
        ridge_mse,
        lasso_mse,
        dt_mse,
        rf_mse
    ],

    "MAE": [
        mean_absolute_error(y_test, ridge_pred),
        mean_absolute_error(y_test, lasso_pred),
        dt_mae,
        rf_mae
    ],

    "RMSE": [
        np.sqrt(ridge_mse),
        np.sqrt(lasso_mse),
        np.sqrt(dt_mse),
        np.sqrt(rf_mse)
    ],

    "R2": [
        ridge_r2,
        lasso_r2,
        dt_r2,
        rf_r2
    ]
})

print(comparison)

           Model Type          Model           MSE           MAE  \
0  Regularized Linear          Ridge  6.608583e+12  9.243632e+06   
1  Regularized Linear          Lasso  6.607015e+12  9.244905e+06   
2          Tree-Based  Decision Tree  9.298332e+12  2.346348e+06   
3          Tree-Based  Random Forest  5.604729e+12  1.772108e+06   

           RMSE        R2  
0  2.570716e+06  0.917942  
1  2.570411e+06  0.917961  
2  3.049317e+06  0.878880  
3  2.367431e+06  0.926993  


**Final Conclusion**

The Real Estate dataset was analyzed and prepared to predict House Price using different machine learning regression techniques. The dataset was divided into features and target variables, followed by a train-test split and appropriate preprocessing.

Several regression approaches were implemented, including Ridge Regression, Lasso Regression, Decision Tree Regression, Random Forest Regression, and Support Vector Regression (SVR). Feature scaling was applied where required, especially for linear and SVR models, while tree-based models were trained without scaling.

Different cross-validation techniques were also studied, including K-Fold Cross-Validation, Stratified K-Fold using target-value binning, Leave-One-Out Cross-Validation, and Time Series Split. These methods demonstrated how different validation strategies can produce different estimates of model performance depending on the structure of the data.

For regularized linear models, Ridge (L2) reduced the magnitude of coefficients, while Lasso (L1) could reduce some coefficients to zero, making it useful for feature selection. Decision Tree Regression was able to capture non-linear relationships, while Random Forest used multiple decision trees to produce an ensemble prediction. SVR was evaluated using different kernels, including Linear, Polynomial, and RBF, along with different values of C, gamma, and epsilon.

All models were evaluated using MSE, MAE, RMSE, and R² Score. MSE, MAE, and RMSE were used to measure prediction error, where lower values indicate lower error, while R² was used to measure how well the model explains variation in house prices.

Final conclusion for report